# Churn Use Case: Medallion Architecture Demo

## Overview

Este notebook demuestra una implementación de la Arquitectura de Medallón (Medallion Architecture) en el Workbench de Oracle AI Data Platform (AIDP), usando un caso de uso para análisis de analíticos de "churn" o "abandono". La arquitectura de medallón organiza los datos en tres capas:

* Capa Bronze (Bronce): Ingestión de datos crudos y su almacenamiento.
* Capa Silver (Plata): Datos limpios, validados y estructurados.
* Capa Gold (Oro): Datos agregaods y listos para análisis con modelos ML.

## Caso de Uso

El caso consiste en analizar los datos de productos usados por los clientes, así como los cambios de uso de los diferentes tiers de productos y la cancelación por parte de los clientes.

## Configuración del entorno en AIDP
Este notebook aprovecha la sesión existente de Spark en el entorno AIDP.

## Ingesta de datos desde la fuente original (Legacy) 

Este notebook usa una biblioteca Python para conectarse directamente a MySQL, lee los datos por lotes y entrega cada lote a Spark para escribirlo como Parquet o CSV en OCI Object Storage.

Para un entorno local puedes instalar `mysql-connector-python` con `%pip install mysql-connector-python`.

## Ingesta desde MySQL a OCI Object Storage

Este notebook usa una biblioteca Python para conectarse directamente a MySQL, lee los datos por lotes y entrega cada lote a Spark para escribirlo como Parquet o CSV en OCI Object Storage.

In [ ]:
# Ejecuta sólo si el entorno no contiene la dependencia.
# %pip install mysql-connector-python

from datetime import datetime
import pytz
import json


In [ ]:
# Pretty Print
from pyspark.sql import DataFrame
from IPython.display import display, HTML
from html import escape

_original_show = DataFrame.show

def _html_show(self, n=20, truncate=True, vertical=False):
    if vertical:
        return _original_show(self, n, truncate, vertical)

    if truncate is True:
        truncate_value = 20
    elif truncate is False:
        truncate_value = 0
    else:
        truncate_value = int(truncate)

    text = self._jdf.showString(
        n,
        truncate_value,
        False
    )

    display(HTML(f"""
    <div style="
        width: 100%;
        max-width: 100%;
        overflow-x: auto;
        border: 1px solid #ddd;
        padding: 8px;
        box-sizing: border-box;
    ">
        <pre style="
            white-space: pre;
            width: max-content;
            min-width: max-content;
            margin: 0;
        ">{escape(text)}</pre>
    </div>
    """))

DataFrame.show = _html_show

In [ ]:
# Este método nos permitirá tomar los valores directo desde los parametros del Job si están disponibles
# Si no lo están usarán la variable de entorno o el valor de Default
import os

def get_config(name, env_name=None, default=""):
    env_name = env_name or name
    try:
        value = oidlUtils.parameters.getParameter(name, "")
        if value:
            return value
    except Exception:
        pass
    return os.getenv(env_name, default)

# Declaración de Variables

TABLES = ["customers", "products", "subscriptions", "billing_payments"]

EXPORT_FORMAT = get_config("export_format", "EXPORT_FORMAT", "parquet").lower()
CHUNK_SIZE = int(get_config("chunk_size", "EXPORT_CHUNK_SIZE", "10000"))
BUCKET_NAME = get_config("oci_bucket", "OCI_OBJECT_STORAGE_BUCKET", "OCI_BUCKET_NAME")
NAMESPACE = get_config("oci_namespace", "OCI_OBJECT_STORAGE_NAMESPACE", "idajmumkp9ca")
OBJECT_PREFIX = get_config("oci_storage_prefix", "OCI_OBJECT_STORAGE_PREFIX", "aidp-workshop/mysql-ingestion").strip("/")
EXTRACTION_ID = datetime.now(pytz.utc).strftime("%Y%m%dT%H%M%SZ")
MYSQL_CONFIG = {
    "host": get_config("mysql_host", "MYSQL_HOST", "IP_ADDRESS"),
    "port": int(get_config("mysql_port", "MYSQL_PORT", "3306")),
    "user": get_config("mysql_user", "MYSQL_USERNAME",  "MYSQL_USER"),
    "password": os.environ.get("MYSQL_PASSWORD", "MYSQL_PASSWORD"),            
    "database": get_config("mysql_db", "MYSQL_DATABASE", "aidp_churn_workshop"),
}
OBJECT_STORAGE_BASE = f"oci://{BUCKET_NAME}@{NAMESPACE}/{OBJECT_PREFIX}"

if EXPORT_FORMAT not in {"parquet", "csv"}:
    raise ValueError("EXPORT_FORMAT debe ser 'parquet' o 'csv'.")
if CHUNK_SIZE <= 0:
    raise ValueError("EXPORT_CHUNK_SIZE debe ser mayor que cero.")

print({"tables": TABLES, "format": EXPORT_FORMAT, "chunk_size": CHUNK_SIZE,
       "destination": OBJECT_STORAGE_BASE, "extraction_id": EXTRACTION_ID})

### Conexión al los datos transaccionales - Fuente Legacy

El caso de uso supone que existe una fuente de datos transaccional, donde se registran los movimientos de clientes y los productos usados por ellos. Para el caso de este taller se trata de un base de datos en MySQL 5.7, se trata de una BD Legacy pero los sistemas de operación de la compañía utilizan esa base de datos y en esa versión para funcionar. 

El sistema de registro de datos operativos podría beneficiarse de una modernización, sin embargo, se ha vuelto necesario poder analizar esos datos en este momento, pero no es recomendable hacerlo usando directamente la base de datos legacy.

Crearemos la manera de comunicarnos con MySQL y extraer los datos para analizarlos en AIDP.

* Definimos los métodos para crear la conexión a MySQL.
* Extraemos los datos y los colocamos en dataframes para volcar su contenido en archivos parquet o csv.
* Crearemos además archivos de registro o auditoria de la extracción.
* Una vez que tengamos los archivos en OCI Object Storage podremos leerlos para integrarlos en la capa bronce.

## Métodos para crear la conexión a MySQL y creación del registro de extracción

In [ ]:
def write_batch(dataframe, target_path, mode):
    writer = dataframe.write.mode(mode)

    if EXPORT_FORMAT == "parquet":
        writer.option("compression", "snappy").parquet(target_path)
    else:
        writer.option("header", "true").option("encoding", "UTF-8").csv(target_path)


def ingest_table(table_name):
    jdbc_url = (
        f"jdbc:mysql://{MYSQL_CONFIG['host']}:"
        f"{MYSQL_CONFIG.get('port', 3306)}/"
        f"{MYSQL_CONFIG['database']}"
        "?sslMode=DISABLED"
    )

    dataframe = (
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", table_name)
        .option("user", MYSQL_CONFIG["user"])
        .option("password", MYSQL_CONFIG["password"])
        .option("driver", "com.mysql.cj.jdbc.Driver")
        .option("fetchsize", CHUNK_SIZE)
        .load()
    )

    target_path = (
        f"{OBJECT_STORAGE_BASE}/"
        f"{table_name}/"
        f"extraction_id={EXTRACTION_ID}"
    )

    rows_read = dataframe.count()

    write_batch(
        dataframe,
        target_path,
        "overwrite"
    )

    print(
        f"{table_name}: "
        f"filas={rows_read:,}; "
        f"destino={target_path}"
    )

    return {
        "table": table_name,
        "path": target_path,
        "rows": rows_read,
    }

In [ ]:
results = [
    ingest_table(table_name)
    for table_name in TABLES
]

manifest = {
    "extraction_id": EXTRACTION_ID,
    "source_database": MYSQL_CONFIG["database"],
    "format": EXPORT_FORMAT,
    "fetch_size": CHUNK_SIZE,
    "created_at_utc": datetime.now(pytz.utc).isoformat(),
    "tables": results,
}

manifest_path = (
    f"{OBJECT_STORAGE_BASE}/"
    f"_manifests/"
    f"extraction_id={EXTRACTION_ID}"
)

manifest_df = spark.createDataFrame(
    [
        (
            json.dumps(
                manifest,
                ensure_ascii=False
            ),
        )
    ],
    ["manifest_json"]
)

manifest_df.coalesce(1).write.mode(
    "errorifexists"
).text(manifest_path)

print(
    f"Ingesta terminada. "
    f"Manifest publicado en {manifest_path}"
)


In [ ]:
# Validación opcional de una salida escrita por Spark.
sample_path = results[0]["path"]

validation_df = spark.read.parquet(sample_path)

validation_df.printSchema()
validation_df.createOrReplaceTempView("validation_sample")


In [ ]:
validation_df.show(5, truncate=False)

In [ ]:
%sql
SELECT *
FROM validation_sample
LIMIT 5

## Configuración: Creación del catálogo ChurnAnalysis y de los esquemas de medallón

Crearemos los siguientes elementos:

* churn_analysis.bronze: Datos de transacción originales.
* churn_analysis.silver: Transacciones limpias y validadas.
* churn_analysis.gold: Datos listos para analíticos y ML.

Esta estructura provee aislamiento y gobierno de los datos a través de las capas del medallón.

In [ ]:
# Create churn catalog and medallion schemas

spark.sql("CREATE CATALOG IF NOT EXISTS churn_analysis")

spark.sql("CREATE SCHEMA IF NOT EXISTS churn_analysis.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS churn_analysis.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS churn_analysis.gold")

print("Churn analysis catalog and medallion schemas created successfully!")
print("- churn_analysis.bronze: Raw transaction data")
print("- churn_analysis.silver: Cleaned and validated data")
print("- churn_analysis.gold: Analytics and ML-ready data")

<div style="
  background-color: #CD7F32;
  color: #FFFFFF;
  padding: 18px 22px;
  border-radius: 8px;
  margin: 12px 0 18px 0;
">
  <h2 style="margin: 0 0 10px 0;">
    Capa Bronce: Ingesta de datos sin procesar
  </h2>

  <h3>Diseño de la capa de bronce</h3>

  <p style="margin: 0; line-height: 1.6;">
    La capa «bronce» almacena los datos brutos de las transacciones a medida que se incorporan, 
    con un procesamiento mínimo. Utilizaremos tablas Delta con agrupamiento «liquid» para obtener un rendimiento óptimo.
  </p>

  <h3>Tabla: `customers_bronze` </h3>

- Registros de clientes sin procesar con todos los campos originales
- Agrupación dinámica por `customer_status` y `cancelled_at`

<h3>Tabla: `products_bronze` </h3>

- Productos registrados sin procesar con todos los campos originales
- Agrupación dinámica por `product_category` y `service_level_rank`

<h3>Tabla: `acquired_products_bronze` </h3>

- Productos contratados por los clientes, sin procesar con todos los campos originales
- Agrupación dinámica por `customer_id` y `subscription_status`

<h3>Tabla: `billing_payments_bronze` </h3>

- Pagos por los productos contratados por los clientes, sin procesar con todos los campos originales
- Agrupación dinámica por `subscription_id` y `due_at`

En todas las tablas de la capa bronce
- Preserva la integridad y la auditabilidad de los datos

</div>

In [ ]:
# Create Bronze Layer Delta tables with liquid clustering

# customers_bronze table
spark.sql("""
CREATE TABLE IF NOT EXISTS churn_analysis.bronze.customers_bronze (
    customer_id INT,
    first_name STRING,
    last_name STRING,
    age INT,
    email STRING,
    phone STRING,
    state_province STRING,
    city STRING,
    customer_status STRING,
    cancelled_at TIMESTAMP ,
    cancellation_type STRING,
    cancellation_reason_code STRING,
    cancellation_reason_detail STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
CLUSTER BY (customer_status, cancelled_at)
""")

print("Bronze layer table customers_bronze created successfully!")
print("Liquid clustering will automatically optimize data layout for customer_status and cancelled_at queries.")

# products_bronze table
spark.sql("""
CREATE TABLE IF NOT EXISTS churn_analysis.bronze.products_bronze (
    product_id INT,
    product_code STRING,
    product_name STRING,
    product_category STRING,
    service_level_rank INT,
    monthly_list_price DECIMAL(12,2),
    is_active BOOLEAN,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
CLUSTER BY (product_category, service_level_rank)
""")

print("Bronze layer table products_bronze created successfully!")
print("Liquid clustering will automatically optimize data layout for product_category and service_level_rank queries.")

# acquired_products_bronze table
spark.sql("""
CREATE TABLE IF NOT EXISTS churn_analysis.bronze.acquired_products_bronze (
    subscription_id INT,
    customer_id INT,
    product_id INT,
    subscription_status STRING,
    subscription_type STRING,
    contracted_price DECIMAL(12,2),
    started_at TIMESTAMP,
    next_billing_at TIMESTAMP,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
CLUSTER BY (customer_id, subscription_status)
""")

print("Bronze layer table acquired_products_bronze created successfully!")
print("Liquid clustering will automatically optimize data layout for customer_id and subscription_status queries.")

# billing_payments_bronze table
spark.sql("""
CREATE TABLE IF NOT EXISTS churn_analysis.bronze.billing_payments_bronze (
    billing_payment_id INT,
    subscription_id INT,
    billing_status STRING,
    due_at TIMESTAMP,
    total_amount DECIMAL(12,2),
    payment_method STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
CLUSTER BY (billing_status, due_at)
""")
    
print("Bronze layer table billing_payments_bronze created successfully!")
print("Liquid clustering will automatically optimize data layout for billing_status and due_at queries.")

In [ ]:
# Cargar los archivos de la extracción y anexarlos a las tablas Delta de la capa Bronze.

from pyspark.sql import functions as F

SOURCE_TO_BRONZE = {
    "customers": "churn_analysis.bronze.customers_bronze",
    "products": "churn_analysis.bronze.products_bronze",
    # La tabla Bronze conserva el nombre histórico acquired_products_bronze;
    # la fuente transaccional se llama subscriptions.
    "subscriptions": "churn_analysis.bronze.acquired_products_bronze",
    "billing_payments": "churn_analysis.bronze.billing_payments_bronze",
}

if "results" not in globals():
    raise RuntimeError("Ejecuta primero la celda de ingesta que crea la variable results.")

results_by_table = {item["table"]: item for item in results}
missing_results = set(SOURCE_TO_BRONZE) - set(results_by_table)
if missing_results:
    raise RuntimeError(f"No hay rutas de extracción para: {sorted(missing_results)}")

def read_exported_table(source_table):
    path = results_by_table[source_table]["path"]
    if EXPORT_FORMAT == "parquet":
        return spark.read.parquet(path)
    return (spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(path))

def conform_to_bronze_schema(dataframe, target_table):
    target_schema = spark.table(target_table).schema
    source_columns = set(dataframe.columns)
    required_columns = {field.name for field in target_schema}
    missing_columns = required_columns - source_columns
    if missing_columns:
        raise ValueError(
            f"La fuente no contiene columnas requeridas por {target_table}: "
            f"{sorted(missing_columns)}"
        )

    # Seleccionar en el orden de la tabla y convertir tipos; esto también
    # permite insertar CSV cuyos timestamps/números fueron inferidos como texto.
    return dataframe.select(*[
        F.col(field.name).cast(field.dataType).alias(field.name)
        for field in target_schema
    ])

inserted_rows = {}
for source_table, target_table in SOURCE_TO_BRONZE.items():
    source_df = read_exported_table(source_table)
    bronze_df = conform_to_bronze_schema(source_df, target_table)
    row_count = bronze_df.count()
    bronze_df.write.mode("append").saveAsTable(target_table)
    inserted_rows[target_table] = row_count
    print(f"{source_table} -> {target_table}: {row_count:,} filas insertadas")

print("Carga de la capa Bronze terminada.")

<div style="
  background-color: #C0C0C0;
  color: #000000;
  padding: 18px 22px;
  border-radius: 8px;
  margin: 12px 0 18px 0;
">
  <h2 style="margin: 0 0 10px 0;">
    Capa Plata: Limpieza y validacion de datos
  </h2>

  <h3>Diseño de la capa de plata</h3>

  <p style="margin: 0; line-height: 1.6;">
    La capa «plata» proporciona datos limpios, validados y enriquecidos. Haremos lo siguiente:
  </p>
  
- Verificar pagos que son extemporáneos
- Agregar un identificador de pagos tardíos
- Identificar cambios en los productos usados por los clientes.

<h3>Tabla: `billing_payments_silver` </h3>

- Datos de pagos limpios con validación de los pagos pasada la fecha que les corresponde
- Listos para el procesamiento analítico

<h3>Tabla: `acquired_products_silver` </h3>

- Datos de productos adquiridos por cada cliente limpios 
- Identificación de movimientos en las categorias de productos 

<h3>Tabla: customers_silver </h3>

- Datos de clientes con consistencia sobre cancelaciones de productos adquiridos

</div>

In [ ]:
# Create Silver Layer Delta tables

# billing_payments_silver table
spark.sql("""
CREATE TABLE IF NOT EXISTS churn_analysis.silver.billing_payments_silver (
    billing_payment_id INT,
    subscription_id INT,
    customer_id INT,
    product_id INT,
    billing_status STRING,
    updated_billing_status STRING,
    due_at TIMESTAMP,
    total_amount DECIMAL(12,2),
    payment_method STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
CLUSTER BY (customer_id, product_id)
""")

print("Silver layer table billing_payments_silver created successfully!")

# acquired_products_silver table
spark.sql("""
CREATE TABLE IF NOT EXISTS churn_analysis.silver.acquired_products_silver (
    subscription_id INT,
    customer_id INT,
    product_id INT,
    subscription_status STRING,
    subscription_type STRING,
    corrected_subscription_type STRING,
    contracted_price DECIMAL(12,2),
    started_at TIMESTAMP,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
CLUSTER BY (customer_id)
""")

print("Silver layer table acquired_products_silver created successfully!")
print("Silver layer tables are ready for data transformation and enrichment.")

# customers_silver table
spark.sql("""
CREATE TABLE IF NOT EXISTS churn_analysis.silver.customers_silver (
    customer_id INT,
    first_name STRING,
    last_name STRING,
    age INT,
    email STRING,
    phone STRING,
    state_province STRING,
    city STRING,
    customer_status STRING,
    cancelled_at TIMESTAMP,
    cancellation_type STRING,
    cancellation_reason_code STRING,
    cancellation_reason_detail STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
CLUSTER BY (customer_status, cancelled_at)
""")

print("Silver layer table customers_silver created successfully!")

La siguiente celda realiza lo siguiente:
* Importa pyspark.sql.functions as F.
* Lee churn_analysis.bronze.billing_payments_bronze.
* Hace un left join con acquired_products_bronze para obtener customer_id y product_id, ya que esos campos existen en la tabla Silver.

Calcula:
* OVERDUE  # created_at > due_at
* INTIME   # created_at <= due_at

Escribe el resultado en:
* churn_analysis.silver.billing_payments_silver

In [ ]:
# Transformación Silver de billing_payments.
from pyspark.sql import functions as F

BRONZE_BILLING_TABLE = "churn_analysis.bronze.billing_payments_bronze"
BRONZE_SUBSCRIPTIONS_TABLE = "churn_analysis.bronze.acquired_products_bronze"
SILVER_BILLING_TABLE = "churn_analysis.silver.billing_payments_silver"

billing_payments_bronze_df = spark.table(BRONZE_BILLING_TABLE)

# La tabla Bronze de pagos sólo contiene subscription_id. Se incorporan
# customer_id y product_id desde la suscripción para completar la tabla Silver.
subscription_lookup_df = (
    spark.table(BRONZE_SUBSCRIPTIONS_TABLE)
    .select("subscription_id", "customer_id", "product_id")
    .dropDuplicates(["subscription_id"])
)

billing_payments_silver_df = (
    billing_payments_bronze_df
    .join(subscription_lookup_df, on="subscription_id", how="left")
    .withColumn(
        "updated_billing_status",
        F.when(
            F.col("created_at") > F.col("due_at"),
            F.lit("OVERDUE")
        ).otherwise(F.lit("INTIME"))
    )
)

# Seleccionar el orden de columnas de la tabla destino y escribir en Silver.
silver_columns = spark.table(SILVER_BILLING_TABLE).columns
missing_columns = set(silver_columns) - set(billing_payments_silver_df.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas por {SILVER_BILLING_TABLE}: {sorted(missing_columns)}")

billing_payments_silver_df = billing_payments_silver_df.select(*silver_columns)
billing_payments_silver_df.write.mode("overwrite").insertInto(SILVER_BILLING_TABLE)

print(f"Tabla {SILVER_BILLING_TABLE} actualizada: {billing_payments_silver_df.count():,} filas")
billing_payments_silver_df.show(5, truncate=False)

La siguiente celda realiza lo siguiente:
* Lee churn_analysis.bronze.acquired_products_bronze.
* Obtiene service_level_rank desde products_bronze.
* Para cada cliente, elimina las suscripciones repetidas al mismo producto, conservando la más antigua.
* Conserva CANCELLED en la última suscripción cuando el cliente ya abandonó; en caso contrario la última queda ACTIVE.
* Marca las suscripciones anteriores como CANCELLED.

<dl>
<dt>Calcula corrected_subscription_type:</dt>
<dd>- DOWNGRADE si el tier anterior era superior.</dd>
<dd>- UPGRADE si el tier anterior era inferior.</dd>
<dd>- NEW para la primera suscripción o cuando el tier es equivalente.</dd>
</dl>

* Escribe el resultado en churn_analysis.silver.acquired_products_silver usando overwrite.

In [ ]:
# Transformación Silver de acquired_products.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_SUBSCRIPTIONS_TABLE = "churn_analysis.bronze.acquired_products_bronze"
BRONZE_PRODUCTS_TABLE = "churn_analysis.bronze.products_bronze"
SILVER_SUBSCRIPTIONS_TABLE = "churn_analysis.silver.acquired_products_silver"

acquired_products_bronze_df = spark.table(BRONZE_SUBSCRIPTIONS_TABLE)
products_lookup_df = (
    spark.table(BRONZE_PRODUCTS_TABLE)
    .select("product_id", "service_level_rank")
)

subscriptions_with_tier_df = acquired_products_bronze_df.join(
    products_lookup_df, on="product_id", how="left"
)

# Si el cliente repite un producto, conservar la primera suscripción
# y descartar la más nueva, porque no representa un cambio real.
# Excepción: siempre se conserva la última suscripción de origen, pues
# puede ser el evento terminal que evidencia la cancelación del cliente.
latest_source_subscription_window = Window.partitionBy("customer_id").orderBy(
    F.col("created_at").desc(), F.col("subscription_id").desc()
)
dedup_window = Window.partitionBy("customer_id", "product_id").orderBy(
    F.col("created_at").asc(), F.col("subscription_id").asc()
)
subscriptions_dedup_df = (
    subscriptions_with_tier_df
    .withColumn("source_latest_order", F.row_number().over(latest_source_subscription_window))
    .withColumn("same_product_order", F.row_number().over(dedup_window))
    .filter((F.col("same_product_order") == 1) | (F.col("source_latest_order") == 1))
    .drop("same_product_order", "source_latest_order")
)

customer_history_window = Window.partitionBy("customer_id").orderBy(
    F.col("created_at").asc(), F.col("subscription_id").asc()
)
latest_subscription_window = Window.partitionBy("customer_id").orderBy(
    F.col("created_at").desc(), F.col("subscription_id").desc()
)

acquired_products_silver_df = (
    subscriptions_dedup_df
    .withColumn("previous_product_id", F.lag("product_id").over(customer_history_window))
    .withColumn("previous_service_level_rank", F.lag("service_level_rank").over(customer_history_window))
    .withColumn("source_subscription_status", F.col("subscription_status"))
    .withColumn("latest_order", F.row_number().over(latest_subscription_window))
    .withColumn(
        "subscription_status",
        F.when(F.col("latest_order") != 1, F.lit("CANCELLED"))
         .when(F.col("source_subscription_status") == F.lit("CANCELLED"), F.lit("CANCELLED"))
         .otherwise(F.lit("ACTIVE"))
    )
    .withColumn(
        "corrected_subscription_type",
        F.when(F.col("previous_product_id").isNull(), F.lit("NEW"))
         .when(
             F.col("previous_service_level_rank") > F.col("service_level_rank"),
             F.lit("DOWNGRADE")
         )
         .when(
             F.col("previous_service_level_rank") < F.col("service_level_rank"),
             F.lit("UPGRADE")
         )
         .otherwise(F.lit("NEW"))
    )
    .drop("previous_product_id", "previous_service_level_rank", "service_level_rank", "source_subscription_status", "latest_order")
)

# Alinear el DataFrame con el esquema de la tabla destino y reemplazar la
# versión Silver anterior por el resultado corregido.
silver_subscription_columns = spark.table(SILVER_SUBSCRIPTIONS_TABLE).columns
missing_columns = set(silver_subscription_columns) - set(acquired_products_silver_df.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas por {SILVER_SUBSCRIPTIONS_TABLE}: {sorted(missing_columns)}")

acquired_products_silver_df = acquired_products_silver_df.select(*silver_subscription_columns)
acquired_products_silver_df.write.mode("overwrite").insertInto(SILVER_SUBSCRIPTIONS_TABLE)

print(f"Tabla {SILVER_SUBSCRIPTIONS_TABLE} actualizada: {acquired_products_silver_df.count():,} filas")
acquired_products_silver_df.show(10, truncate=False)

La celda siguiente realiza lo siguiente:

Lee el último registro de acquired_products_silver por cliente, ordenando por created_at. Si el último estado es CANCELLED, asigna:cancelled_at usando updated_at de la suscripción. cancellation_type = "VOLUNTARY".

Si el último estado no es CANCELLED, ambos campos quedan en NULL. Los clientes sin suscripciones también conservan esos campos en NULL.

El resultado se escribe en churn_analysis.silver.customers_silver con overwrite.

In [ ]:
# Transformación Silver de customers.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_CUSTOMERS_TABLE = "churn_analysis.bronze.customers_bronze"
SILVER_SUBSCRIPTIONS_TABLE = "churn_analysis.silver.acquired_products_silver"
SILVER_CUSTOMERS_TABLE = "churn_analysis.silver.customers_silver"

customers_bronze_df = spark.table(BRONZE_CUSTOMERS_TABLE)
acquired_products_silver_df = spark.table(SILVER_SUBSCRIPTIONS_TABLE)

# La última suscripción de cada cliente se identifica por created_at.
latest_subscription_window = Window.partitionBy("customer_id").orderBy(
    F.col("created_at").desc(), F.col("subscription_id").desc()
)
latest_subscription_df = (
    acquired_products_silver_df
    .withColumn("subscription_order", F.row_number().over(latest_subscription_window))
    .filter(F.col("subscription_order") == 1)
    # updated_at representa el momento en que se actualizó el estado a CANCELLED.
    .select("customer_id", "subscription_status", "updated_at")
    .withColumnRenamed("subscription_status", "latest_subscription_status")
    .withColumnRenamed("updated_at", "subscription_cancelled_at")
)

customers_silver_df = (
    customers_bronze_df
    .join(latest_subscription_df, on="customer_id", how="left")
    # La cancelación explícita del cliente en Bronze es la fuente autoritativa.
    # La última suscripción CANCELLED complementa esa evidencia cuando el
    # sistema legado no haya poblado los campos del cliente.
    .withColumn(
        "is_cancelled",
        (F.col("customer_status") == F.lit("CLOSED"))
        | (F.col("latest_subscription_status") == F.lit("CANCELLED"))
    )
    .withColumn(
        "cancelled_at",
        F.when(
            F.col("is_cancelled"),
            F.coalesce(F.col("cancelled_at"), F.col("subscription_cancelled_at"))
        ).otherwise(F.lit(None).cast("timestamp"))
    )
    .withColumn(
        "cancellation_type",
        F.when(
            F.col("is_cancelled"),
            F.coalesce(F.col("cancellation_type"), F.lit("VOLUNTARY"))
        ).otherwise(F.lit(None).cast("string"))
    )
    .withColumn(
        "customer_status",
        F.when(F.col("is_cancelled"), F.lit("CLOSED")).otherwise(F.col("customer_status"))
    )
    .drop("latest_subscription_status", "subscription_cancelled_at", "is_cancelled")
)

# Mantener el esquema de customers_silver y reemplazar la versión anterior.
silver_customer_columns = spark.table(SILVER_CUSTOMERS_TABLE).columns
missing_columns = set(silver_customer_columns) - set(customers_silver_df.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas por {SILVER_CUSTOMERS_TABLE}: {sorted(missing_columns)}")

customers_silver_df = customers_silver_df.select(*silver_customer_columns)
customers_silver_df.write.mode("overwrite").insertInto(SILVER_CUSTOMERS_TABLE)

print(f"Tabla {SILVER_CUSTOMERS_TABLE} actualizada: {customers_silver_df.count():,} filas")
customers_silver_df.show(10, truncate=False)

<div style="
  background-color: #f2c22d;
  color: #000000;
  padding: 18px 22px;
  border-radius: 8px;
  margin: 12px 0 18px 0;
">
  <h2 style="margin: 0 0 10px 0;">
    Capa Oro: análisis y datos preparados para el aprendizaje automático
  </h2>

  <h3>Diseño de la capa de oro</h3>

  <p style="margin: 0; line-height: 1.6;">
    La capa «Oro» proporciona funciones de análisis y aprendizaje automático listas para su uso empresarial. Crearemos:
  </p>
  
- Tablas de análisis agregadas
- Ingeniería de características preparadas para ML
- Entrenamiento de modelos de predicción de churn

<h3>Tablas de la capa oro: </h3>

- `acquired_products_analytics_gold`: vista Customer 360 y métricas agregadas de producto y pagos
- `customer_churn_features_gold`: snapshots históricos de características para entrenar churn
- `churn_predictions_gold`: resultados de scoring, segmento de riesgo y versión del modelo

</div>

In [ ]:
# Tablas Delta de la capa Oro.
spark.sql("""
CREATE TABLE IF NOT EXISTS churn_analysis.gold.acquired_products_analytics_gold (
    as_of_date DATE,
    customer_id INT,
    age INT,
    state_province STRING,
    customer_status STRING,
    cancelled_at TIMESTAMP,
    current_subscription_id INT,
    current_product_id INT,
    current_product_name STRING,
    current_product_category STRING,
    current_service_level_rank INT,
    current_contracted_price DECIMAL(12,2),
    current_subscription_started_at TIMESTAMP,
    days_in_current_product INT,
    subscriptions_lifetime INT,
    upgrades_lifetime INT,
    downgrades_lifetime INT,
    days_since_last_product_change INT,
    billing_payments_90d INT,
    overdue_payments_90d INT,
    overdue_ratio_90d DOUBLE,
    days_since_last_overdue INT,
    total_billed_90d DECIMAL(18,2),
    average_billed_90d DECIMAL(18,2),
    latest_payment_method STRING,
    latest_payment_due_at TIMESTAMP,
    created_at TIMESTAMP
)
USING DELTA
CLUSTER BY (as_of_date, customer_id)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS churn_analysis.gold.customer_churn_features_gold (
    as_of_date DATE,
    customer_id INT,
    age INT,
    state_province STRING,
    current_product_category STRING,
    current_service_level_rank INT,
    current_contracted_price DECIMAL(12,2),
    customer_tenure_days INT,
    days_in_current_product INT,
    subscriptions_lifetime INT,
    upgrades_lifetime INT,
    downgrades_lifetime INT,
    days_since_last_product_change INT,
    billing_payments_90d INT,
    overdue_payments_90d INT,
    overdue_ratio_90d DOUBLE,
    days_since_last_overdue INT,
    total_billed_90d DECIMAL(18,2),
    average_billed_90d DECIMAL(18,2),
    latest_payment_method STRING,
    label_churn_30d INT,
    is_label_mature BOOLEAN,
    created_at TIMESTAMP
)
USING DELTA
CLUSTER BY (as_of_date, customer_id)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS churn_analysis.gold.churn_predictions_gold (
    scored_at TIMESTAMP,
    as_of_date DATE,
    customer_id INT,
    churn_probability DOUBLE,
    predicted_churn INT,
    risk_segment STRING,
    model_name STRING,
    model_version STRING
)
USING DELTA
CLUSTER BY (as_of_date, customer_id)
""")

print("Tablas Gold creadas correctamente.")

### Customer 360 y métricas de riesgo

La siguiente transformación combina el perfil del cliente, su última suscripción, sus movimientos de tier y los pagos de los últimos 90 días. El resultado permite analizar clientes con atrasos recurrentes, downgrades y valor facturado en riesgo sin incluir información de churn futuro.

In [ ]:
# Customer 360: métricas actuales de producto y pagos.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CUSTOMERS_SILVER = "churn_analysis.silver.customers_silver"
SUBSCRIPTIONS_SILVER = "churn_analysis.silver.acquired_products_silver"
PRODUCTS_BRONZE = "churn_analysis.bronze.products_bronze"
PAYMENTS_SILVER = "churn_analysis.silver.billing_payments_silver"
ANALYTICS_GOLD = "churn_analysis.gold.acquired_products_analytics_gold"

customers_df = spark.table(CUSTOMERS_SILVER)
subscriptions_df = spark.table(SUBSCRIPTIONS_SILVER)
products_df = spark.table(PRODUCTS_BRONZE).select(
    "product_id", "product_name", "product_category", "service_level_rank"
)
payments_df = spark.table(PAYMENTS_SILVER)

subscriptions_enriched_df = subscriptions_df.join(products_df, on="product_id", how="left")
latest_subscription_window = Window.partitionBy("customer_id").orderBy(
    F.col("created_at").desc(), F.col("subscription_id").desc()
)
latest_subscription_df = (
    subscriptions_enriched_df
    .withColumn("subscription_order", F.row_number().over(latest_subscription_window))
    .filter(F.col("subscription_order") == 1)
    .select(
        "customer_id",
        F.col("subscription_id").alias("current_subscription_id"),
        F.col("product_id").alias("current_product_id"),
        F.col("product_name").alias("current_product_name"),
        F.col("product_category").alias("current_product_category"),
        F.col("service_level_rank").alias("current_service_level_rank"),
        F.col("contracted_price").alias("current_contracted_price"),
        F.col("started_at").alias("current_subscription_started_at"),
        F.col("created_at").alias("last_product_change_at")
    )
)

subscription_metrics_df = (
    subscriptions_df.groupBy("customer_id").agg(
        F.countDistinct("subscription_id").cast("int").alias("subscriptions_lifetime"),
        F.sum(F.when(F.col("corrected_subscription_type") == "UPGRADE", 1).otherwise(0)).cast("int").alias("upgrades_lifetime"),
        F.sum(F.when(F.col("corrected_subscription_type") == "DOWNGRADE", 1).otherwise(0)).cast("int").alias("downgrades_lifetime")
    )
)

payments_90d_df = payments_df.filter(
    (F.to_date("due_at") <= F.current_date())
    & (F.to_date("due_at") > F.date_sub(F.current_date(), 90))
)
payment_metrics_df = (
    payments_90d_df.groupBy("customer_id").agg(
        F.count("billing_payment_id").cast("int").alias("billing_payments_90d"),
        F.sum(F.when(F.col("updated_billing_status") == "OVERDUE", 1).otherwise(0)).cast("int").alias("overdue_payments_90d"),
        F.max(F.when(F.col("updated_billing_status") == "OVERDUE", F.to_date("due_at"))).alias("last_overdue_due_at"),
        F.sum("total_amount").cast("decimal(18,2)").alias("total_billed_90d"),
        F.avg("total_amount").cast("decimal(18,2)").alias("average_billed_90d")
    )
)
latest_payment_window = Window.partitionBy("customer_id").orderBy(
    F.col("due_at").desc(), F.col("billing_payment_id").desc()
)
latest_payment_df = (
    payments_df
    .withColumn("payment_order", F.row_number().over(latest_payment_window))
    .filter(F.col("payment_order") == 1)
    .select(
        "customer_id",
        F.col("payment_method").alias("latest_payment_method"),
        F.col("due_at").alias("latest_payment_due_at")
    )
)

analytics_gold_df = (
    customers_df
    .select("customer_id", "age", "state_province", "customer_status", "cancelled_at")
    .join(latest_subscription_df, on="customer_id", how="left")
    .join(subscription_metrics_df, on="customer_id", how="left")
    .join(payment_metrics_df, on="customer_id", how="left")
    .join(latest_payment_df, on="customer_id", how="left")
    .withColumn("as_of_date", F.current_date())
    .withColumn("days_in_current_product", F.datediff(F.current_date(), F.to_date("current_subscription_started_at")).cast("int"))
    .withColumn("days_since_last_product_change", F.datediff(F.current_date(), F.to_date("last_product_change_at")).cast("int"))
    .withColumn("days_since_last_overdue", F.datediff(F.current_date(), F.col("last_overdue_due_at")).cast("int"))
    .withColumn("overdue_ratio_90d", F.when(F.col("billing_payments_90d") > 0, F.col("overdue_payments_90d") / F.col("billing_payments_90d")).otherwise(F.lit(0.0)))
    .fillna({
        "age": -1, "current_service_level_rank": 0, "current_contracted_price": 0.0,
        "days_in_current_product": -1, "days_since_last_product_change": -1,
        "days_since_last_overdue": -1,
        "subscriptions_lifetime": 0, "upgrades_lifetime": 0, "downgrades_lifetime": 0,
        "billing_payments_90d": 0, "overdue_payments_90d": 0, "overdue_ratio_90d": 0.0,
        "total_billed_90d": 0.0, "average_billed_90d": 0.0
    })
    .drop("last_product_change_at", "last_overdue_due_at")
    .withColumn("created_at", F.current_timestamp())
)

analytics_columns = spark.table(ANALYTICS_GOLD).columns
analytics_gold_df = analytics_gold_df.select(*analytics_columns)
analytics_gold_df.write.mode("overwrite").insertInto(ANALYTICS_GOLD)

print(f"Tabla {ANALYTICS_GOLD} actualizada: {analytics_gold_df.count():,} clientes")
analytics_gold_df.show(10, truncate=False)

### Visualización de salud de clientes

Las siguientes gráficas usan `acquired_products_analytics_gold`. Spark calcula las agregaciones, sólo se recopilan los resultados agregados necesarios para mostrar los gráficos.

In [ ]:
# Indicadores generales y distribución de riesgo de pagos.
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

analytics_visual_df = spark.table("churn_analysis.gold.acquired_products_analytics_gold")
kpis = analytics_visual_df.agg(
    F.countDistinct("customer_id").alias("total_customers"),
    F.sum(F.when(F.col("overdue_payments_90d") > 0, 1).otherwise(0)).alias("customers_with_overdue"),
    F.avg("overdue_ratio_90d").alias("average_overdue_ratio"),
    F.sum("total_billed_90d").alias("total_billed_90d")
).first().asDict()

risk_bucket_df = (
    analytics_visual_df
    .withColumn(
        "overdue_risk_bucket",
        F.when(F.col("overdue_ratio_90d") == 0, F.lit("Sin atraso"))
         .when(F.col("overdue_ratio_90d") <= 0.33, F.lit("Atraso bajo"))
         .when(F.col("overdue_ratio_90d") <= 0.66, F.lit("Atraso medio"))
         .otherwise(F.lit("Atraso alto"))
    )
    .groupBy("overdue_risk_bucket")
    .agg(F.countDistinct("customer_id").alias("customers"))
)
bucket_order = ["Sin atraso", "Atraso bajo", "Atraso medio", "Atraso alto"]
risk_counts = {row["overdue_risk_bucket"]: row["customers"] for row in risk_bucket_df.collect()}

total_customers = int(kpis["total_customers"] or 0)
customers_with_overdue = int(kpis["customers_with_overdue"] or 0)
average_overdue_ratio = float(kpis["average_overdue_ratio"] or 0.0)
total_billed_90d = float(kpis["total_billed_90d"] or 0.0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(["Total", "Con atraso"], [total_customers, customers_with_overdue], color=["#4C78A8", "#E45756"])
axes[0].set_title("Clientes con pagos vencidos")
axes[0].set_ylabel("Clientes")

axes[1].bar(bucket_order, [risk_counts.get(bucket, 0) for bucket in bucket_order], color=["#72B7B2", "#F2CF5B", "#F58518", "#E45756"])
axes[1].set_title("Distribución de riesgo por atrasos")
axes[1].set_ylabel("Clientes")
axes[1].tick_params(axis="x", rotation=20)

axes[2].axis("off")
axes[2].set_title("Indicadores de cobranza")
axes[2].text(0.5, 0.65, f"{average_overdue_ratio:.1%}", ha="center", fontsize=24, color="#B279A2", fontweight="bold")
axes[2].text(0.5, 0.53, "Atraso promedio (90 días)", ha="center")
axes[2].text(0.5, 0.28, f"${total_billed_90d:,.0f}", ha="center", fontsize=24, color="#54A24B", fontweight="bold")
axes[2].text(0.5, 0.16, "Facturación acumulada (90 días)", ha="center")

fig.suptitle("Customer 360: salud de pagos", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Riesgo por tier y movimientos de producto

Esta vista compara el atraso promedio de cada combinación de familia y nivel de productos con los movimientos acumulados de upgrade y downgrade. Es útil para identificar productos o niveles de servicio que podrían necesitar una estrategia de retención específica.

In [ ]:
# Riesgo de pago y movimientos acumulados por tier.
tier_metrics_df = (
    analytics_visual_df
    .filter(F.col("current_product_category").isNotNull())
    .groupBy("current_product_category", "current_service_level_rank")
    .agg(
        F.countDistinct("customer_id").alias("customers"),
        F.avg("overdue_ratio_90d").alias("average_overdue_ratio"),
        F.avg("upgrades_lifetime").alias("average_upgrades"),
        F.avg("downgrades_lifetime").alias("average_downgrades")
    )
    .orderBy("current_product_category", "current_service_level_rank")
)
tier_rows = tier_metrics_df.collect()
tier_labels = [f"{row['current_product_category']} | tier {row['current_service_level_rank']}" for row in tier_rows]
overdue_values = [float(row['average_overdue_ratio'] or 0.0) * 100 for row in tier_rows]
upgrade_values = [float(row['average_upgrades'] or 0.0) for row in tier_rows]
downgrade_values = [float(row['average_downgrades'] or 0.0) for row in tier_rows]
positions = list(range(len(tier_labels)))

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
axes[0].bar(positions, overdue_values, color="#E45756")
axes[0].set_xticks(positions, tier_labels, rotation=35, ha="right")
axes[0].set_ylabel("% de pagos OVERDUE")
axes[0].set_title("Atraso promedio por tier")

bar_width = 0.38
axes[1].bar([position - bar_width / 2 for position in positions], upgrade_values, width=bar_width, label="Upgrades", color="#4C78A8")
axes[1].bar([position + bar_width / 2 for position in positions], downgrade_values, width=bar_width, label="Downgrades", color="#F58518")
axes[1].set_xticks(positions, tier_labels, rotation=35, ha="right")
axes[1].set_ylabel("Promedio de movimientos por cliente")
axes[1].set_title("Movimientos históricos por tier")
axes[1].legend()

fig.suptitle("Producto, movimiento y riesgo de cobranza", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Snapshots históricos para churn

Un modelo no debe aprender del estado final del cliente. Por eso cada fila de entrenamiento representa un cliente en una fecha de corte (`as_of_date`), usando sólo pagos y suscripciones existentes hasta ese momento. La etiqueta vale 1 si el cliente cancela en los siguientes 30 días. Las razones de cancelación, `cancelled_at` y el estado final se excluyen de las características.

In [ ]:
# Construcción de snapshots de características y etiqueta churn_30d.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ML_HORIZON_DAYS = 30
PAYMENT_LOOKBACK_DAYS = 90
FEATURES_GOLD = "churn_analysis.gold.customer_churn_features_gold"

customer_source_df = spark.table("churn_analysis.silver.customers_silver").select(
    "customer_id", "age", "state_province",
    F.to_date("created_at").alias("customer_created_date"),
    F.to_date("cancelled_at").alias("customer_cancelled_date")
)
product_source_df = spark.table("churn_analysis.bronze.products_bronze").select(
    "product_id", "product_category", "service_level_rank"
)
subscription_source_df = (
    spark.table("churn_analysis.silver.acquired_products_silver")
    .join(product_source_df, on="product_id", how="left")
    .select(
        "customer_id", "subscription_id", "product_id", "contracted_price",
        "corrected_subscription_type", "product_category", "service_level_rank",
        F.to_date("created_at").alias("subscription_created_date")
    )
)
payment_source_df = spark.table("churn_analysis.silver.billing_payments_silver").select(
    "customer_id", "billing_payment_id", "total_amount", "payment_method",
    "updated_billing_status", F.to_date("due_at").alias("due_date")
)

# Las fechas de vencimiento proporcionan cortes históricos para cada cliente.
snapshot_base_df = (
    payment_source_df.select("customer_id", F.col("due_date").alias("as_of_date"))
    .filter(F.col("as_of_date").isNotNull())
    .distinct()
    .join(customer_source_df, on="customer_id", how="inner")
    .filter(F.col("customer_created_date") <= F.col("as_of_date"))
    .filter(F.col("customer_cancelled_date").isNull() | (F.col("customer_cancelled_date") > F.col("as_of_date")))
)

max_observed_date = payment_source_df.agg(F.max("due_date").alias("max_date")).first()["max_date"]
if max_observed_date is None:
    raise ValueError("No hay fechas due_at para construir snapshots de churn.")

snapshot_base_df = (
    snapshot_base_df
    .withColumn("is_label_mature", F.col("as_of_date") <= F.date_sub(F.lit(max_observed_date), ML_HORIZON_DAYS))
    .withColumn(
        "label_churn_30d",
        F.when(
            (F.col("customer_cancelled_date") > F.col("as_of_date"))
            & (F.col("customer_cancelled_date") <= F.date_add(F.col("as_of_date"), ML_HORIZON_DAYS)),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
)

snapshot_keys_df = snapshot_base_df.select("customer_id", "as_of_date")
payment_history_df = (
    snapshot_keys_df.alias("s")
    .join(
        payment_source_df.alias("p"),
        (F.col("s.customer_id") == F.col("p.customer_id"))
        & (F.col("p.due_date") <= F.col("s.as_of_date"))
        & (F.col("p.due_date") > F.date_sub(F.col("s.as_of_date"), PAYMENT_LOOKBACK_DAYS)),
        how="left"
    )
    .select(
        F.col("s.customer_id").alias("customer_id"), F.col("s.as_of_date").alias("as_of_date"),
        F.col("p.billing_payment_id").alias("billing_payment_id"), F.col("p.total_amount").alias("total_amount"),
        F.col("p.payment_method").alias("payment_method"), F.col("p.updated_billing_status").alias("updated_billing_status"),
        F.col("p.due_date").alias("due_date")
    )
)

payment_feature_df = (
    payment_history_df.groupBy("customer_id", "as_of_date").agg(
        F.count("billing_payment_id").cast("int").alias("billing_payments_90d"),
        F.sum(F.when(F.col("updated_billing_status") == "OVERDUE", 1).otherwise(0)).cast("int").alias("overdue_payments_90d"),
        F.max(F.when(F.col("updated_billing_status") == "OVERDUE", F.col("due_date"))).alias("last_overdue_date"),
        F.sum("total_amount").cast("decimal(18,2)").alias("total_billed_90d"),
        F.avg("total_amount").cast("decimal(18,2)").alias("average_billed_90d")
    )
    .withColumn("overdue_ratio_90d", F.when(F.col("billing_payments_90d") > 0, F.col("overdue_payments_90d") / F.col("billing_payments_90d")).otherwise(F.lit(0.0)))
)

payment_snapshot_window = Window.partitionBy("customer_id", "as_of_date").orderBy(
    F.col("due_date").desc(), F.col("billing_payment_id").desc()
)
latest_payment_feature_df = (
    payment_history_df
    .withColumn("payment_order", F.row_number().over(payment_snapshot_window))
    .filter(F.col("payment_order") == 1)
    .select("customer_id", "as_of_date", F.col("payment_method").alias("latest_payment_method"))
)

subscription_history_df = (
    snapshot_keys_df.alias("s")
    .join(
        subscription_source_df.alias("p"),
        (F.col("s.customer_id") == F.col("p.customer_id"))
        & (F.col("p.subscription_created_date") <= F.col("s.as_of_date")),
        how="left"
    )
    .select(
        F.col("s.customer_id").alias("customer_id"), F.col("s.as_of_date").alias("as_of_date"),
        F.col("p.subscription_id").alias("subscription_id"), F.col("p.contracted_price").alias("contracted_price"),
        F.col("p.corrected_subscription_type").alias("corrected_subscription_type"),
        F.col("p.product_category").alias("product_category"), F.col("p.service_level_rank").alias("service_level_rank"),
        F.col("p.subscription_created_date").alias("subscription_created_date")
    )
)

subscription_feature_df = (
    subscription_history_df.groupBy("customer_id", "as_of_date").agg(
        F.count("subscription_id").cast("int").alias("subscriptions_lifetime"),
        F.sum(F.when(F.col("corrected_subscription_type") == "UPGRADE", 1).otherwise(0)).cast("int").alias("upgrades_lifetime"),
        F.sum(F.when(F.col("corrected_subscription_type") == "DOWNGRADE", 1).otherwise(0)).cast("int").alias("downgrades_lifetime")
    )
)

subscription_snapshot_window = Window.partitionBy("customer_id", "as_of_date").orderBy(
    F.col("subscription_created_date").desc(), F.col("subscription_id").desc()
)
latest_subscription_feature_df = (
    subscription_history_df
    .withColumn("subscription_order", F.row_number().over(subscription_snapshot_window))
    .filter(F.col("subscription_order") == 1)
    .select(
        "customer_id", "as_of_date",
        F.col("product_category").alias("current_product_category"),
        F.col("service_level_rank").cast("int").alias("current_service_level_rank"),
        F.col("contracted_price").cast("decimal(12,2)").alias("current_contracted_price"),
        F.col("subscription_created_date").alias("last_product_change_date")
    )
)

churn_features_df = (
    snapshot_base_df
    .join(payment_feature_df, on=["customer_id", "as_of_date"], how="left")
    .join(latest_payment_feature_df, on=["customer_id", "as_of_date"], how="left")
    .join(subscription_feature_df, on=["customer_id", "as_of_date"], how="left")
    .join(latest_subscription_feature_df, on=["customer_id", "as_of_date"], how="left")
    .withColumn("customer_tenure_days", F.datediff(F.col("as_of_date"), F.col("customer_created_date")).cast("int"))
    .withColumn("days_in_current_product", F.datediff(F.col("as_of_date"), F.col("last_product_change_date")).cast("int"))
    .withColumn("days_since_last_product_change", F.datediff(F.col("as_of_date"), F.col("last_product_change_date")).cast("int"))
    .withColumn("days_since_last_overdue", F.datediff(F.col("as_of_date"), F.col("last_overdue_date")).cast("int"))
    .fillna({
        "age": -1, "current_service_level_rank": 0, "current_contracted_price": 0.0,
        "customer_tenure_days": -1, "days_in_current_product": -1,
        "days_since_last_product_change": -1, "days_since_last_overdue": -1,
        "subscriptions_lifetime": 0, "upgrades_lifetime": 0, "downgrades_lifetime": 0,
        "billing_payments_90d": 0, "overdue_payments_90d": 0, "overdue_ratio_90d": 0.0,
        "total_billed_90d": 0.0, "average_billed_90d": 0.0
    })
    .drop("customer_created_date", "customer_cancelled_date", "last_overdue_date", "last_product_change_date")
    .withColumn("created_at", F.current_timestamp())
)

feature_columns = spark.table(FEATURES_GOLD).columns
churn_features_df = churn_features_df.select(*feature_columns)
churn_features_df.write.mode("overwrite").insertInto(FEATURES_GOLD)

print(f"Tabla {FEATURES_GOLD} actualizada: {churn_features_df.count():,} snapshots")
churn_features_df.groupBy("label_churn_30d", "is_label_mature").count().show()

### Entrenamiento y evaluación temporal

Se usa una regresión logística como línea base explicable. La separación es temporal: los snapshots más antiguos entrenan el modelo y los más recientes lo evalúan. El entrenamiento requiere al menos dos fechas maduras y ejemplos de churn positivos y negativos.

In [ ]:
# Pipeline de ML: baseline de churn con separación temporal.
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import Imputer, OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

FEATURES_GOLD = "churn_analysis.gold.customer_churn_features_gold"
MODEL_NAME = "churn_logistic_regression"
MODEL_VERSION = "v1"

numeric_features = [
    "age", "current_service_level_rank", "current_contracted_price",
    "customer_tenure_days", "days_in_current_product", "subscriptions_lifetime",
    "upgrades_lifetime", "downgrades_lifetime", "days_since_last_product_change",
    "billing_payments_90d", "overdue_payments_90d", "overdue_ratio_90d",
    "days_since_last_overdue", "total_billed_90d", "average_billed_90d"
]
categorical_features = ["state_province", "current_product_category", "latest_payment_method"]

def prepare_model_input(dataframe):
    prepared_df = dataframe
    for column in numeric_features:
        prepared_df = prepared_df.withColumn(column, F.col(column).cast("double"))
    return prepared_df

model_source_df = prepare_model_input(
    spark.table(FEATURES_GOLD).filter(F.col("is_label_mature"))
)
all_snapshot_dates = [row["as_of_date"] for row in spark.table(FEATURES_GOLD).select("as_of_date").distinct().orderBy("as_of_date").collect()]
available_dates = [row["as_of_date"] for row in model_source_df.select("as_of_date").distinct().orderBy("as_of_date").collect()]
ml_training_ready = False
churn_pipeline_model = None
test_predictions_df = None

if len(available_dates) < 2:
    print(
        "Entrenamiento omitido: se requieren al menos dos fechas de corte maduras para una validación temporal. "
        f"Se encontraron {len(available_dates)} fecha(s) madura(s) de {len(all_snapshot_dates)} fecha(s) total(es): "
        f"{[str(date) for date in available_dates]}"
    )
    print("Genera o conserva snapshots de al menos dos fechas due_at separadas en el tiempo y vuelve a construir la tabla Gold.")
else:
    split_date = available_dates[max(0, int(len(available_dates) * 0.8) - 1)]
    train_df = model_source_df.filter(F.col("as_of_date") <= F.lit(split_date))
    test_df = model_source_df.filter(F.col("as_of_date") > F.lit(split_date))
    train_label_counts = {row["label_churn_30d"]: row["count"] for row in train_df.groupBy("label_churn_30d").count().collect()}
    positive_count = train_label_counts.get(1, 0)
    negative_count = train_label_counts.get(0, 0)

    if test_df.limit(1).count() == 0:
        print("Entrenamiento omitido: no hay snapshots posteriores al corte de entrenamiento para evaluar el modelo.")
    elif positive_count == 0 or negative_count == 0:
        print(
            "Entrenamiento omitido: el periodo de entrenamiento requiere ejemplos maduros de churn y no churn. "
            f"Churn: {positive_count}; no churn: {negative_count}."
        )
    else:
        positive_weight = float(negative_count) / float(positive_count)
        train_df = train_df.withColumn("class_weight", F.when(F.col("label_churn_30d") == 1, F.lit(positive_weight)).otherwise(F.lit(1.0)))

        imputed_numeric_features = [f"{column}_imputed" for column in numeric_features]
        indexed_categorical_features = [f"{column}_index" for column in categorical_features]
        encoded_categorical_features = [f"{column}_ohe" for column in categorical_features]
        imputer = Imputer(inputCols=numeric_features, outputCols=imputed_numeric_features).setStrategy("median")
        indexers = [StringIndexer(inputCol=column, outputCol=f"{column}_index", handleInvalid="keep") for column in categorical_features]
        encoder = OneHotEncoder(inputCols=indexed_categorical_features, outputCols=encoded_categorical_features, handleInvalid="keep")
        assembler = VectorAssembler(inputCols=imputed_numeric_features + encoded_categorical_features, outputCol="features", handleInvalid="keep")
        classifier = LogisticRegression(
            featuresCol="features", labelCol="label_churn_30d", weightCol="class_weight",
            maxIter=100, regParam=0.05
        )

        churn_pipeline = Pipeline(stages=[imputer] + indexers + [encoder, assembler, classifier])
        churn_pipeline_model = churn_pipeline.fit(train_df)
        test_predictions_df = churn_pipeline_model.transform(test_df).withColumn(
            "churn_probability", vector_to_array(F.col("probability"))[1]
        )

        roc_auc = BinaryClassificationEvaluator(
            labelCol="label_churn_30d", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
        ).evaluate(test_predictions_df)
        pr_auc = BinaryClassificationEvaluator(
            labelCol="label_churn_30d", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
        ).evaluate(test_predictions_df)
        test_row_count = test_predictions_df.count()
        top_k = max(1, int(test_row_count * 0.10))
        precision_at_10pct = (
            test_predictions_df.orderBy(F.col("churn_probability").desc()).limit(top_k)
            .agg(F.avg("label_churn_30d").alias("precision"))
            .first()["precision"]
        )
        ml_training_ready = True

        print({
            "train_until": str(split_date), "positive_train_weight": positive_weight,
            "roc_auc": roc_auc, "pr_auc": pr_auc, "precision_at_10pct": precision_at_10pct
        })
        test_predictions_df.select("customer_id", "as_of_date", "label_churn_30d", "churn_probability", "prediction").show(20, truncate=False)

### Scoring y activación

El modelo se aplica al corte más reciente disponible. La tabla de resultados conserva la fecha de scoring y la versión del modelo, de modo que un dashboard o una campaña de retención pueda priorizar a los clientes de riesgo alto.

In [ ]:
# Scoring del corte más reciente y publicación de resultados.
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

PREDICTIONS_GOLD = "churn_analysis.gold.churn_predictions_gold"

if not globals().get("ml_training_ready", False) or churn_pipeline_model is None:
    print("Scoring omitido: el modelo no fue entrenado porque aún no se cumplen las validaciones de datos históricos.")
else:
    latest_scoring_date = spark.table(FEATURES_GOLD).agg(F.max("as_of_date").alias("as_of_date")).first()["as_of_date"]
    scoring_features_df = prepare_model_input(
        spark.table(FEATURES_GOLD).filter(F.col("as_of_date") == F.lit(latest_scoring_date))
    )

    churn_scores_df = (
        churn_pipeline_model.transform(scoring_features_df)
        .withColumn("churn_probability", vector_to_array(F.col("probability"))[1])
        .withColumn("predicted_churn", F.col("prediction").cast("int"))
        .withColumn(
            "risk_segment",
            F.when(F.col("churn_probability") >= 0.70, F.lit("HIGH"))
             .when(F.col("churn_probability") >= 0.40, F.lit("MEDIUM"))
             .otherwise(F.lit("LOW"))
        )
        .select(
            F.current_timestamp().alias("scored_at"),
            "as_of_date", "customer_id", "churn_probability", "predicted_churn", "risk_segment",
            F.lit(MODEL_NAME).alias("model_name"),
            F.lit(MODEL_VERSION).alias("model_version")
        )
    )

    churn_scores_df.write.mode("append").insertInto(PREDICTIONS_GOLD)
    print(f"Scoring terminado para {latest_scoring_date}: {churn_scores_df.count():,} clientes")
    churn_scores_df.orderBy(F.col("churn_probability").desc()).show(20, truncate=False)

### Visualización de clientes con mayor riesgo de churn

Después del scoring, esta vista toma la predicción más reciente de cada cliente y estima el ingreso mensual esperado en riesgo como `churn_probability × current_contracted_price`. Es una estimación de priorización para retención, no una proyección financiera definitiva.

In [ ]:
# Clientes de mayor riesgo e impacto financiero mensual estimado.
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

PREDICTIONS_GOLD = "churn_analysis.gold.churn_predictions_gold"
ANALYTICS_GOLD = "churn_analysis.gold.acquired_products_analytics_gold"

latest_score_window = Window.partitionBy("customer_id").orderBy(F.col("scored_at").desc())
latest_scores_df = (
    spark.table(PREDICTIONS_GOLD)
    .withColumn("score_order", F.row_number().over(latest_score_window))
    .filter(F.col("score_order") == 1)
    .drop("score_order")
)
if latest_scores_df.limit(1).count() == 0:
    raise ValueError("No hay resultados de scoring. Ejecuta primero la celda de scoring.")

financial_risk_df = (
    latest_scores_df
    .join(
        spark.table(ANALYTICS_GOLD).select("customer_id", "current_contracted_price", "current_product_category"),
        on="customer_id",
        how="left"
    )
    .withColumn("monthly_contract_value", F.coalesce(F.col("current_contracted_price").cast("double"), F.lit(0.0)))
    .withColumn("expected_monthly_revenue_at_risk", F.col("churn_probability") * F.col("monthly_contract_value"))
)

segment_metrics_df = (
    financial_risk_df.groupBy("risk_segment").agg(
        F.countDistinct("customer_id").alias("customers"),
        F.sum("monthly_contract_value").alias("monthly_contract_value"),
        F.sum("expected_monthly_revenue_at_risk").alias("expected_monthly_revenue_at_risk")
    )
)
segment_rows = {row["risk_segment"]: row.asDict() for row in segment_metrics_df.collect()}
segment_order = ["LOW", "MEDIUM", "HIGH"]
segment_colors = ["#72B7B2", "#F2CF5B", "#E45756"]
customer_counts = [int(segment_rows.get(segment, {}).get("customers") or 0) for segment in segment_order]
expected_revenue = [float(segment_rows.get(segment, {}).get("expected_monthly_revenue_at_risk") or 0.0) for segment in segment_order]

top_risk_rows = (
    financial_risk_df
    .select("customer_id", "churn_probability", "monthly_contract_value", "expected_monthly_revenue_at_risk", "current_product_category")
    .orderBy(F.col("churn_probability").desc(), F.col("expected_monthly_revenue_at_risk").desc())
    .limit(15)
    .collect()
)

top_customer_labels = [f"Cliente {row['customer_id']}" for row in top_risk_rows]
top_probabilities = [float(row['churn_probability']) * 100 for row in top_risk_rows]
top_expected_revenue = [float(row['expected_monthly_revenue_at_risk'] or 0.0) for row in top_risk_rows]

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
axes[0].bar(segment_order, customer_counts, color=segment_colors)
axes[0].set_title("Clientes por segmento de riesgo")
axes[0].set_ylabel("Clientes")
for position, value in enumerate(customer_counts):
    axes[0].text(position, value, str(value), ha="center", va="bottom")

axes[1].bar(segment_order, expected_revenue, color=segment_colors)
axes[1].set_title("Ingreso mensual esperado en riesgo")
axes[1].set_ylabel("Monto estimado")
for position, value in enumerate(expected_revenue):
    axes[1].text(position, value, f"${value:,.0f}", ha="center", va="bottom")

axes[2].barh(top_customer_labels[::-1], top_probabilities[::-1], color="#E45756")
axes[2].set_title("15 clientes con mayor probabilidad de churn")
axes[2].set_xlabel("Probabilidad de churn (%)")
axes[2].set_xlim(0, 100)
for position, (probability, revenue) in enumerate(zip(top_probabilities[::-1], top_expected_revenue[::-1])):
    axes[2].text(probability + 1, position, f"${revenue:,.0f}", va="center", fontsize=8)

high_risk_count = customer_counts[2]
high_risk_expected_revenue = expected_revenue[2]
fig.suptitle(
    f"Churn: {high_risk_count} clientes de riesgo alto | ${high_risk_expected_revenue:,.0f} de ingreso mensual esperado en riesgo",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.show()

financial_risk_df.orderBy(F.col("churn_probability").desc()).select(
    "customer_id", "current_product_category", "churn_probability",
    "monthly_contract_value", "expected_monthly_revenue_at_risk", "risk_segment"
).show(20, truncate=False)